#1: Imports and Path Setup

In [1]:
import os
import sys
import numpy as np
import tensorflow as tf

In [2]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Transformer_Mine

Mounted at /content/drive
/content/drive/MyDrive/Transformer_Mine


In [4]:
from multi_head_attention import MultiHeadAttention
from feed_forward import FeedForwardNetwork
from positional_encoding import PositionalEncoding
from encoder import Encoder
from data_pipeline import (
    create_padding_mask,
    create_look_ahead_mask,
    create_decoder_mask,
)

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

TensorFlow: 2.20.0
Project root: /content/drive/MyDrive/Transformer_Mine


# 2: Mathematical Foundation

From the paper (Section 3.1):

Each decoder layer has **THREE sub-layers**:

### 1. Masked Multi-Head Self-Attention

- Q = K = V = decoder input so far
- Mask: combined causal + padding mask
- Prevents attending to future target tokens

### 2. Multi-Head Cross-Attention (Encoder-Decoder Attention)

- Q = decoder state (from sub-layer 1)
- K = V = encoder output
- Mask: padding mask on encoder input
- Allows decoder to attend to all encoder positions

### 3. Position-wise Feed-Forward Network

- Same as encoder FFN

Each sub-layer wrapped with residual + LayerNorm (POST-LN):

```text
# Sub-layer 1: masked self-attention
attn1_out = MaskedMHA(x, x, x, dec_mask)
x         = LayerNorm(x + Dropout(attn1_out))

# Sub-layer 2: cross-attention
attn2_out = MHA(x, enc_out, enc_out, enc_mask)
x         = LayerNorm(x + Dropout(attn2_out))

# Sub-layer 3: feed-forward
ffn_out   = FFN(x)
x         = LayerNorm(x + Dropout(ffn_out))

```
### Full decoder stack
tgt_ids -> Embedding -> PositionalEncoding
        -> [N x DecoderLayer(enc_output, masks)]
        -> decoder output: (batch, tgt_len, d_model)

In [5]:
print("Decoder architecture (Transformer Base):")
print()
print("  tgt_ids")
print("     |")
print("  Embedding (vocab_size, d_model=512)")
print("     |")
print("  PositionalEncoding (dropout=0.1)")
print("     |")
print("  [DecoderLayer x 6]")
print("     |-- Masked MultiHeadAttention (causal + pad mask)")
print("     |-- Add + LayerNorm")
print("     |-- Cross MultiHeadAttention (enc_output, enc_mask)")
print("     |-- Add + LayerNorm")
print("     |-- FeedForwardNetwork")
print("     |-- Add + LayerNorm")
print("     |")
print("  decoder output: (batch, tgt_len, 512)")

Decoder architecture (Transformer Base):

  tgt_ids
     |
  Embedding (vocab_size, d_model=512)
     |
  PositionalEncoding (dropout=0.1)
     |
  [DecoderLayer x 6]
     |-- Masked MultiHeadAttention (causal + pad mask)
     |-- Add + LayerNorm
     |-- Cross MultiHeadAttention (enc_output, enc_mask)
     |-- Add + LayerNorm
     |-- FeedForwardNetwork
     |-- Add + LayerNorm
     |
  decoder output: (batch, tgt_len, 512)


# 3: DecoderLayer

One decoder layer = three sub-layers, each with residual  
connection and layer normalization (POST-LN, original paper).

In [7]:
class DecoderLayer(tf.keras.layers.Layer):
    """
    Single decoder layer.

    Architecture (POST-LN, original paper):
        x = LayerNorm(x + MaskedMHA(x, x, x, dec_mask))
        x = LayerNorm(x + CrossMHA(x, enc_out, enc_out, enc_mask))
        x = LayerNorm(x + FFN(x))

    Args:
        d_model : model dimension (e.g. 512)
        num_heads : number of attention heads (e.g. 8)
        d_ff : feed-forward inner dimension (e.g. 2048)
        dropout : dropout rate (e.g. 0.1)
    """

    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1, **kwargs):
        super().__init__(**kwargs)

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropot_rate = dropout
        self.supports_masking = True

        # Sub-layer 1: masked self-attention
        # Q = K = V = decoder input
        # Uses causal + padding mask to prevent future token access
        self.self_attention = MultiHeadAttention(
            d_model = d_model,
            num_heads = num_heads,
            name = 'decoder_self_attention',
        )

        # Sub-layer 2: cross-attention (encoder-decoder attention)
        # Q = decoder state, K = V = encoder output
        self.cross_attention = MultiHeadAttention(
            d_model = d_model,
            num_heads = num_heads,
            name = 'decoder_cross_attention',
        )

        # Sub-layer 3: feed-forward
        self.ffn = FeedForwardNetwork(
            d_model = d_model,
            d_ff = d_ff,
            dropout = dropout,
            name = 'decoder_ffn',
        )

        # Layer normalization after each sub-layer
        self.norm_1 = tf.keras.layers.LayerNormalization(
            epsilon = 1e-6,
            name = 'decoder_norm_1',
        )

        self.norm_2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6, name='decoder_norm_2'
        )

        self.norm_3 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6, name='decoder_norm_3'
        )

        # Dropout for each sub-layer output
        self.dropout_1 = tf.keras.layers.Dropout(dropout)

        self.dropout_2 = tf.keras.layers.Dropout(dropout)

        self.dropout_3 = tf.keras.layers.Dropout(dropout)

    def call(self,x,enc_output,dec_mask=None,enc_mask=None,training=False,
    ):
        """
        Forward pass of one decoder layer.

        Args:
            x          : decoder input (batch, tgt_len, d_model)
            enc_output : encoder output (batch, src_len, d_model)
            dec_mask   : combined causal + padding mask
                         shape (batch, 1, tgt_len, tgt_len)
            enc_mask   : encoder padding mask
                         shape (batch, 1, 1, src_len)
            training   : controls dropout

        Returns:
            x               : (batch, tgt_len, d_model)
            self_attn_w     : (batch, num_heads, tgt_len, tgt_len)
            cross_attn_w    : (batch, num_heads, tgt_len, src_len)
        """

        # Sub-layer 1: masked self-attention
        # Q = K = V = x (decoder attends to itself)
        # dec_mask prevents attending to future positions
        self_attn_out, self_attn_w = self.self_attention(
            x, x, x,
            mask=dec_mask,
            training=training,
        )
        # self_attn_out shape: (batch, tgt_len, d_model)

        self_attn_out = self.dropout_1(self_attn_out, training=training)
        x = self.norm_1(x + self_attn_out)
        # x shape: (batch, tgt_len, d_model)

        # Sub-layer 2: cross-attention
        # Q = decoder state (x), K = V = encoder output
        # enc_mask prevents attending to PAD positions in encoder
        cross_attn_out, cross_attn_w = self.cross_attention(
            x, enc_output, enc_output,
            mask=enc_mask,
            training=training,
        )
        # cross_attn_out shape: (batch, tgt_len, d_model)
        # cross_attn_w   shape: (batch, num_heads, tgt_len, src_len)
        cross_attn_out = self.dropout_2(cross_attn_out, training=training)
        x = self.norm_2(x + cross_attn_out)

         # Sub-layer 3: feed-forward network
        ffn_out = self.ffn(x, training=training)
        # ffn_out shape: (batch, tgt_len, d_model)
        ffn_out = self.dropout_3(ffn_out, training=training)
        x = self.norm_3(x + ffn_out)
        # x shape: (batch, tgt_len, d_model)

        return x, self_attn_w, cross_attn_w

    def get_config(self):
        config = super().get_config()

        config.update({
            'd_model' : self.d_model,
            'num_heads': self.num_heads,
            'd_ff' : self.d_ff,
            'dropout' : self.dropout_rate,
        })

        return config

#4: Decoder Stack

In [28]:
class Decoder(tf.keras.layers.Layer):
    """
    Full decoder stack.

    tgt_ids -> Embedding -> PositionalEncoding
            -> [DecoderLayer x N]
            -> decoder output: (batch, tgt_len, d_model)

    Args:
        num_layers : number of decoder layers (paper: 6)
        d_model : model dimension (paper: 512)
        num_heads : attention heads (paper: 8)
        d_ff : FFN inner dim (paper: 2048)
        vocab_size : target vocabulary size
        max_len : max sequence length for positional encoding
        dropout : dropout rate (paper: 0.1)
    """

    def __init__(self, num_layers, d_model, num_heads, vocab_size, d_ff, max_len = 5000, dropout = 0.1, **kwargs,):
        super().__init__(**kwargs)

        self.num_layers = num_layers
        self.d_model = d_model
        self.d_ff = d_ff # Store d_ff
        self.supports_masking = True

        # Token embedding for target sequence
        self.embedding = tf.keras.layers.Embedding(
            input_dim = vocab_size,
            output_dim = d_model,
            name = 'decoder_embedding',
        )

        # Positional encoding (includes embedding scaling + dropout)
        self.pos_encoding = PositionalEncoding(
            d_model = d_model,
            max_len = max_len,
            dropout = dropout,
            name = 'decoder_pos_encoding',
        )

        # Stack of N decoder layers
        self.dec_layers = [
            DecoderLayer(
                d_model = d_model,
                num_heads = num_heads,
                d_ff = d_ff,
                dropout = dropout,
                name = f'decoder_layer_{i}',
            )

            for i in range(num_layers)
        ]

    def call(self, tgt_ids, enc_output, dec_mask=None, enc_mask=None, training=False,):
        """
        Forward pass of the full decoder.

        Args:
            tgt_ids : target token IDs (batch, tgt_len)
            enc_output : encoder output (batch, src_len, d_model)
            dec_mask : combined causal + padding mask
            enc_mask : encoder padding mask
            training : controls dropout

        Returns:
            x : (batch, tgt_len, d_model)
            attention_weights: dict with self and cross attn per layer
        """

        x = self.embedding(tgt_ids)
        # x shape: (batch, tgt_len, d_model)

        x = self.pos_encoding(x, training=training)
        # x shape: (batch, tgt_len, d_model)

        attention_weights = {}
        for i, layer in enumerate(self.dec_layers):
            x, self_attn_w, cross_attn_w = layer(
                x,
                enc_output,
                dec_mask=dec_mask,
                enc_mask=enc_mask,
                training=training,
            )

            attention_weights[f'decoder_layer_{i}_self']  = self_attn_w

            attention_weights[f'decoder_layer_{i}_cross'] = cross_attn_w
        # x shape: (batch, tgt_len, d_model)

        return x, attention_weights

    def get_config(self):
        config = super().get_config()

        config.update({
            'num_layers' : self.num_layers,
            'd_model' : self.d_model,
            'd_ff' : self.d_ff,
        })

        return config

#5: Shape Tests

In [16]:
NUM_LAYERS = 6
D_MODEL = 512
NUM_HEADS  = 8
D_FF = 2048
VOCAB_SIZE = 8000
BATCH = 2
SRC_LEN = 15
TGT_LEN = 10

In [17]:
tf.random.set_seed(42)

In [18]:
tgt_ids = tf.constant([
    [1, 5, 12, 8, 33, 7, 2, 0, 0, 0],
    [1, 3, 18, 6, 44, 9, 11, 22, 2, 0],
], dtype=tf.int32)

In [20]:
src_ids = tf.constant([
    [5, 12, 8, 33, 7, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [3, 18, 6, 44, 9, 11, 22, 4, 2, 0, 0, 0, 0, 0, 0],
], dtype=tf.int32)


In [21]:
enc_output_sim = tf.random.normal((BATCH, SRC_LEN, D_MODEL))

In [22]:
dec_mask = create_decoder_mask(tgt_ids, tgt_len=TGT_LEN)
enc_mask = create_padding_mask(src_ids)

In [24]:
dec_mask.shape, enc_mask.shape

(TensorShape([2, 1, 10, 10]), TensorShape([2, 1, 1, 15]))

In [29]:
decoder = Decoder(
    num_layers=NUM_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    vocab_size=VOCAB_SIZE,
    dropout=0.0,
)

In [30]:
dec_output, attn_weights = decoder(
    tgt_ids,
    enc_output_sim,
    dec_mask=dec_mask,
    enc_mask=enc_mask,
    training=False,
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [31]:
assert dec_output.shape == (BATCH, TGT_LEN, D_MODEL), \
    f"Decoder output shape wrong : {dec_output.shape}"

print(f"\ninput tgt_ids shape : {tgt_ids.shape}")
print(f"enc_output shape : {enc_output_sim.shape}")
print(f"decoder output shape: {dec_output.shape}")

for key, w in attn_weights.items():
    if 'self' in key:
        expected = (BATCH, NUM_HEADS, TGT_LEN, TGT_LEN)
    else:
        expected = (BATCH, NUM_HEADS, TGT_LEN, SRC_LEN)
    assert w.shape == expected, f"{key} shape wrong: {w.shape}"
    print(f"  {key}: {w.shape}")

print("All shape tests passed.")


input tgt_ids shape : (2, 10)
enc_output shape : (2, 15, 512)
decoder output shape: (2, 10, 512)
  decoder_layer_0_self: (2, 8, 10, 10)
  decoder_layer_0_cross: (2, 8, 10, 15)
  decoder_layer_1_self: (2, 8, 10, 10)
  decoder_layer_1_cross: (2, 8, 10, 15)
  decoder_layer_2_self: (2, 8, 10, 10)
  decoder_layer_2_cross: (2, 8, 10, 15)
  decoder_layer_3_self: (2, 8, 10, 10)
  decoder_layer_3_cross: (2, 8, 10, 15)
  decoder_layer_4_self: (2, 8, 10, 10)
  decoder_layer_4_cross: (2, 8, 10, 15)
  decoder_layer_5_self: (2, 8, 10, 10)
  decoder_layer_5_cross: (2, 8, 10, 15)
All shape tests passed.


# 6: Causal Mask Verification

The most critical decoder property: position i must not  
attend to positions j > i in self-attention.

Verified across all layers and all heads.

In [32]:
tf.random.set_seed(3)

In [33]:
BATCH_C = 2
TGT_C = 8
SRC_C = 10

In [34]:
enc_out_c = tf.random.normal((BATCH_C, SRC_C, D_MODEL))
tgt_ids_c = tf.constant(
    [[1, 5, 8, 3, 7, 2, 0, 0],
     [1, 9, 4, 6, 2, 0, 0, 0]],
    dtype=tf.int32,
)

In [35]:
dec_mask_c = create_decoder_mask(tgt_ids_c, tgt_len=TGT_C)
enc_mask_c = None

In [36]:
dec_causal = Decoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE, dropout=0.0,
)

In [37]:
_, attn_c = dec_causal(
    tgt_ids_c, enc_out_c,
    dec_mask=dec_mask_c,
    enc_mask=enc_mask_c,
    training=False,
)

In [38]:
total_violations = 0

for layer_idx in range(2):
    key = f'decoder_layer_{layer_idx}_self'
    w_np = attn_c[key].numpy()

    for b in range(BATCH_C):
        for h in range(NUM_HEADS):
            for i in range(TGT_C):
                for j in range(i + 1, TGT_C):
                    if w_np[b, h, i, j] > 1e-7:
                        total_violations += 1
                        print(f"violation: layer={layer_idx} "
                              f"batch={b} head={h} "
                              f"query={i} -> key={j} "
                              f"weight={w_np[b,h,i,j]:.6f}")

if total_violations == 0:
    print(f"No causal violations found.")
    print(f"Checked 2 layers x {BATCH_C} batches x "
          f"{NUM_HEADS} heads x {TGT_C} positions.")
    print("Decoder self-attention is correctly causal.")
else:
    print(f"{total_violations} causal violations found.")
    print("Check that dec_mask is being passed to self_attention correctly.")

No causal violations found.
Checked 2 layers x 2 batches x 8 heads x 8 positions.
Decoder self-attention is correctly causal.


# 7: Cross-Attention Verification

Verifies that cross-attention weights have the correct shape  
(tgt_len x src_len) and that encoder PAD positions receive  
~0 attention weight when enc_mask is applied.

In [39]:
tf.random.set_seed(7)

In [40]:
BATCH_X = 2
TGT_X = 6
SRC_X = 10

In [41]:
src_ids_x = tf.constant([
    [5, 12, 8, 33, 7,  0,  0,  0,  0,  0],
    [3, 18, 6, 44, 9, 11, 22,  0,  0,  0],
], dtype=tf.int32)

enc_out_x = tf.random.normal((BATCH_X, SRC_X, D_MODEL))
tgt_ids_x = tf.constant(
    [[1, 5, 8, 3, 7, 2],
     [1, 9, 4, 6, 2, 0]],
    dtype=tf.int32,
)

In [42]:
dec_mask_x = create_decoder_mask(tgt_ids_x, tgt_len=TGT_X)
enc_mask_x = create_padding_mask(src_ids_x)

In [43]:
dec_cross = Decoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE, dropout=0.0,
)

In [44]:
_, attn_x = dec_cross(
    tgt_ids_x, enc_out_x,
    dec_mask=dec_mask_x,
    enc_mask=enc_mask_x,
    training=False,
)

In [45]:
cross_w = attn_x['decoder_layer_1_cross'].numpy()

In [46]:
print(f"Cross-attention weight shape: {cross_w.shape}")
print(f"rows = tgt_len ({TGT_X}), cols = src_len ({SRC_X})")
assert cross_w.shape == (BATCH_X, NUM_HEADS, TGT_X, SRC_X)
print("Cross-attention shape correct.")

pad_cross_w_b0 = cross_w[0, :, :, 5:].max()
print(f"\nBatch 0 encoder PAD positions (5-9):")
print(f"  max cross-attention weight at PAD keys: {pad_cross_w_b0:.2e}")
assert pad_cross_w_b0 < 1e-7
print("encoder PAD positions correctly ignored in cross-attention.")

pad_cross_w_b1 = cross_w[1, :, :, 7:].max()
print(f"\nBatch 1 encoder PAD positions (7-9):")
print(f"max cross-attention weight at PAD keys: {pad_cross_w_b1:.2e}")
assert pad_cross_w_b1 < 1e-7
print("encoder PAD positions correctly ignored in cross-attention.")

print("\nCross-attention verification passed.")

Cross-attention weight shape: (2, 8, 6, 10)
rows = tgt_len (6), cols = src_len (10)
Cross-attention shape correct.

Batch 0 encoder PAD positions (5-9):
  max cross-attention weight at PAD keys: 0.00e+00
encoder PAD positions correctly ignored in cross-attention.

Batch 1 encoder PAD positions (7-9):
max cross-attention weight at PAD keys: 0.00e+00
encoder PAD positions correctly ignored in cross-attention.

Cross-attention verification passed.


# 8: Encoder-Decoder Integration Test

Runs encoder and decoder together end-to-end using  
real token IDs and correct mask generation.

This is the first time the full enc-dec pipeline runs.

In [47]:
tf.random.set_seed(99)


In [48]:
BATCH_I = 2
SRC_LEN_I = 12
TGT_LEN_I = 8
VOCAB_SIZE_I = 8000

In [49]:
src_ids_i = tf.constant([
    [5, 12, 8, 33, 7, 2, 0, 0, 0, 0, 0, 0],
    [3, 18, 6, 44, 9, 11, 22, 4, 2, 0, 0, 0],
], dtype=tf.int32)

tgt_ids_i = tf.constant([
    [1, 15, 22, 8, 2, 0, 0, 0],
    [1, 9,  4,  6, 3, 7, 2, 0],
], dtype=tf.int32)

In [50]:
enc_mask_i = create_padding_mask(src_ids_i)
dec_mask_i = create_decoder_mask(tgt_ids_i, tgt_len=TGT_LEN_I)

In [51]:
 encoder_i = Encoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE_I, dropout=0.0,
)

decoder_i = Decoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE_I, dropout=0.0,
)


In [52]:
# Encoder forward pass
enc_out_i, enc_attn_i = encoder_i(
    src_ids_i, mask=enc_mask_i, training=False
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [54]:
# Decoder forward pass
dec_out_i, dec_attn_i = decoder_i(
    tgt_ids_i, enc_out_i,
    dec_mask=dec_mask_i,
    enc_mask=enc_mask_i,
    training=False,
)

In [55]:
print(f"src_ids shape : {src_ids_i.shape}")
print(f"tgt_ids shape : {tgt_ids_i.shape}")
print(f"enc_output shape : {enc_out_i.shape}")
print(f"dec_output shape : {dec_out_i.shape}")

assert enc_out_i.shape == (BATCH_I, SRC_LEN_I, D_MODEL)
assert dec_out_i.shape == (BATCH_I, TGT_LEN_I, D_MODEL)

print("Encoder attention layers:", len(enc_attn_i))
print("Decoder attention layers:", len(dec_attn_i),
      "(self + cross per layer)")
print("Encoder-Decoder integration test passed.")

src_ids shape : (2, 12)
tgt_ids shape : (2, 8)
enc_output shape : (2, 12, 512)
dec_output shape : (2, 8, 512)
Encoder attention layers: 2
Decoder attention layers: 4 (self + cross per layer)
Encoder-Decoder integration test passed.


#9: Gradient Flow Test

In [56]:
tf.random.set_seed(42)

In [57]:
enc_out_g = tf.random.normal((2, 10, D_MODEL))
tgt_ids_g = tf.constant(
    [[1, 5, 8, 3, 7, 2, 0, 0],
     [1, 9, 4, 6, 2, 0, 0, 0]],
    dtype=tf.int32,
)
dec_mask_g = create_decoder_mask(tgt_ids_g, tgt_len=8)
enc_mask_g = None

dec_grad = Decoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE, dropout=0.0,
)

In [58]:
with tf.GradientTape() as tape:
    out_g, _ = dec_grad(
        tgt_ids_g, enc_out_g,
        dec_mask=dec_mask_g,
        enc_mask=enc_mask_g,
        training=True,
    )
    loss_g = tf.reduce_sum(out_g)

grads = tape.gradient(loss_g, dec_grad.trainable_variables)

none_grads = [v.name for v, g in zip(dec_grad.trainable_variables, grads)
              if g is None]
nan_grads  = [v.name for v, g in zip(dec_grad.trainable_variables, grads)
              if g is not None and tf.reduce_any(tf.math.is_nan(g)).numpy()]
zero_grads = [v.name for v, g in zip(dec_grad.trainable_variables, grads)
              if g is not None and tf.norm(g).numpy() == 0]

print(f"Total parameter tensors : {len(dec_grad.trainable_variables)}")
print(f"None gradients : {len(none_grads)}")
print(f"NaN  gradients : {len(nan_grads)}")
print(f"Zero gradients : {len(zero_grads)}")

if none_grads:
    print("\nParameters with None gradient:")
    for name in none_grads:
        print(f"  {name}")

if nan_grads:
    print("\nParameters with NaN gradient:")
    for name in nan_grads:
        print(f"  {name}")

if not none_grads and not nan_grads:
    print("\nGradient flow test passed.")
    print("All parameters receive valid gradients.")
else:
    print("\nGradient flow test FAILED.")

Total parameter tensors : 37
None gradients : 0
NaN  gradients : 0
Zero gradients : 0

Gradient flow test passed.
All parameters receive valid gradients.


#10: Save Decoder to Drive

In [59]:
DECODER_PY = os.path.join(PROJECT_ROOT, 'decoder.py')

In [60]:
code = '''"""
decoder.py - Transformer Decoder
Transformer Reproduction Project - Phase 9

Usage:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from decoder import DecoderLayer, Decoder
"""

import tensorflow as tf
from multi_head_attention import MultiHeadAttention
from feed_forward import FeedForwardNetwork
from positional_encoding import PositionalEncoding

class DecoderLayer(tf.keras.layers.Layer):
    """
    Single decoder layer (POST-LN, original paper).

    x = LayerNorm(x + MaskedMHA(x, x, x, dec_mask))
    x = LayerNorm(x + CrossMHA(x, enc_out, enc_out, enc_mask))
    x = LayerNorm(x + FFN(x))
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropout_rate = dropout
        self.supports_masking = True

        self.self_attention = MultiHeadAttention(d_model, num_heads,
                                                   name="decoder_self_attention")
        self.cross_attention = MultiHeadAttention(d_model, num_heads,
                                                   name="decoder_cross_attention")
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout,
                                        name="decoder_ffn")
        self.norm_1 = tf.keras.layers.LayerNormalization(epsilon=1e-6,
                                                        name="decoder_norm_1")
        self.norm_2 = tf.keras.layers.LayerNormalization(epsilon=1e-6,
                                                        name="decoder_norm_2")
        self.norm_3 = tf.keras.layers.LayerNormalization(epsilon=1e-6,
                                                        name="decoder_norm_3")

        self.dropout_1 = tf.keras.layers.Dropout(dropout)
        self.dropout_2 = tf.keras.layers.Dropout(dropout)
        self.dropout_3 = tf.keras.layers.Dropout(dropout)

    def call(self, x, enc_output, dec_mask=None, enc_mask=None, training=False):
        self_attn_out, self_attn_w = self.self_attention(
            x, x, x, mask=dec_mask, training=training)
        self_attn_out = self.dropout_1(self_attn_out, training=training)
        x = self.norm_1(x + self_attn_out)

        cross_attn_out, cross_attn_w = self.cross_attention(
            x, enc_output, enc_output, mask=enc_mask, training=training)
        cross_attn_out = self.dropout_2(cross_attn_out, training=training)
        x = self.norm_2(x + cross_attn_out)

        ffn_out = self.ffn(x, training=training)
        ffn_out = self.dropout_3(ffn_out, training=training)
        x = self.norm_3(x + ffn_out)

        return x, self_attn_w, cross_attn_w

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model, "num_heads": self.num_heads,
                        "d_ff": self.d_ff, "dropout": self.dropout_rate})
        return config


class Decoder(tf.keras.layers.Layer):
    """
    Full decoder stack.
    tgt_ids -> Embedding -> PositionalEncoding -> [DecoderLayer x N]
    """

    def __init__(self, num_layers, d_model, num_heads, d_ff,
                 vocab_size, max_len=5000, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_layers = num_layers
        self.d_model = d_model
        self.supports_masking = True
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model,
                                                    name="decoder_embedding")
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout,
                                               name="decoder_pos_encoding")
        self.dec_layers   = [
            DecoderLayer(d_model, num_heads, d_ff, dropout,
                         name=f"decoder_layer_{i}")
            for i in range(num_layers)
        ]

    def call(self, tgt_ids, enc_output, dec_mask=None,
             enc_mask=None, training=False):
        x = self.embedding(tgt_ids)
        x = self.pos_encoding(x, training=training)
        attention_weights = {}
        for i, layer in enumerate(self.dec_layers):
            x, self_attn_w, cross_attn_w = layer(
                x, enc_output,
                dec_mask=dec_mask,
                enc_mask=enc_mask,
                training=training,
            )

            attention_weights[f"decoder_layer_{i}_self"]  = self_attn_w
            attention_weights[f"decoder_layer_{i}_cross"] = cross_attn_w

        return x, attention_weights

    def get_config(self):
        config = super().get_config()
        config.update({"num_layers": self.num_layers, "d_model": self.d_model})
        return config
'''



In [61]:
with open(DECODER_PY, 'w', encoding='utf-8') as f:
    f.write(code)

In [63]:
import importlib
import decoder
importlib.reload(decoder)
from decoder import Decoder as Decoder_loaded

dec_reload = Decoder_loaded(
    num_layers=6, d_model=512, num_heads=8,
    d_ff=2048, vocab_size=8000, dropout=0.0,
)

enc_out_r  = tf.random.normal((1, 10, 512))
tgt_ids_r  = tf.constant([[1, 5, 8, 3, 7, 2, 0, 0]], dtype=tf.int32)
dec_mask_r = create_decoder_mask(tgt_ids_r, tgt_len=8)

out_r, attn_r = dec_reload(
    tgt_ids_r, enc_out_r,
    dec_mask=dec_mask_r,
    training=False,
)

assert out_r.shape == (1, 8, 512)
assert len(attn_r) == 12   # 6 layers x 2 (self + cross)

print("decoder.py saved and reloaded.")
print(f"path : {DECODER_PY}")
print(f"output shape : {out_r.shape}")
print(f"attention entries : {len(attn_r)}  (6 layers x self + cross)")

decoder.py saved and reloaded.
path : /content/drive/MyDrive/Transformer_Mine/decoder.py
output shape : (1, 8, 512)
attention entries : 12  (6 layers x self + cross)
